## Obtención de Detalles en Español desde TMDB

El dataset de películas que tenemos hasta ahora (`movies_final.csv`) contiene información en **inglés** descargada de TMDB. Sin embargo, nuestro público objetivo es principalmente hispanohablante (México), por lo que necesitamos obtener los campos de texto en **español**.

En este notebook descargamos los siguientes campos en español para las 5,000 películas del catálogo:

- **`title`**: título localizado al español
- **`overview`**: sinopsis en español
- **`tagline`**: eslogan en español
- **`genres`**: nombres de géneros en español

Los demás campos (director, actores, keywords, score, poster_path, etc.) se obtendrán del archivo en inglés mediante un merge con `movies_final.csv`.

**Nota:** TMDB no siempre tiene traducción disponible para todos los campos. En esos casos, el campo quedará vacío o conservará el valor en inglés.

In [1]:
import pandas as pd
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import warnings
warnings.filterwarnings('ignore')

import os
from dotenv import load_dotenv
# Load the environment variables .env
load_dotenv()

True

### Carga del Dataset de Películas

Cargamos el dataset de películas para obtener los `tmdb_id` que usaremos para consultar la API.

In [2]:
mdf = pd.read_csv('../data/processed/movies_final.csv')
mdf.head()

,movielens_id,tmdb_id,title,genres,overview,runtime,release_date,tagline,vote_count,vote_average,poster_path,backdrop_path,cast,director,keywords,score,movie_id
0,10,710,GoldenEye,"['Adventure', 'Action', 'Thriller']",When a powerful satellite system falls into th...,130,1995-11-16,No limits. No fears. No substitutes.,4278,6.900,/z0ljRnNxIO7CRBhLEO0DvLgAFPR.jpg,/fIWsCpYR9iGDMSbMTSAzy8L7Kg5.jpg,"['Pierce Brosnan', 'Sean Bean', 'Izabella Scor...",Martin Campbell,"['computer virus', 'cuba', 'falsely accused', ...",6.904242,0
1,6,949,Heat,"['Crime', 'Drama', 'Action']",Obsessive master thief Neil McCauley leads a t...,170,1995-12-15,A Los Angeles crime saga.,8224,7.931,/umSVjVdbVwtx5ryCA2QXL44Durm.jpg,/xKsnZDERG1dk95wuZ5q9iks3OL3.jpg,"['Al Pacino', 'Robert De Niro', 'Val Kilmer']",Michael Mann,"['robbery', 'chase', 'obsession', 'detective',...",7.925015,1
2,1,862,Toy Story,"['Family', 'Comedy', 'Animation', 'Adventure']","Led by Woody, Andy's toys live happily in his ...",81,1995-11-22,The adventure takes off when toys come to life!,19723,7.971,/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,/3Rfvhy1Nl6sSGJwyjb0QiZzZYlB.jpg,"['Tom Hanks', 'Tim Allen', 'Don Rickles']",John Lasseter,"['rescue', 'friendship', 'mission', 'jealousy'...",7.968359,2
3,2,8844,Jumanji,"['Adventure', 'Fantasy', 'Family']",When siblings Judy and Peter discover an encha...,104,1995-12-15,It's a jungle in here.,11207,7.200,/vgpXmVaVyUL7GGiDeiK1mKEKzcX.jpg,/qSxeCfWUUyht9hZgaaYmtPtTkw2.jpg,"['Robin Williams', 'Kirsten Dunst', 'Bradley P...",Joe Johnston,"['giant insect', 'board game', 'disappearance'...",7.199878,3
4,11,9087,The American President,"['Drama', 'Romance']","Widowed U.S. president Andrew Shepherd, one of...",113,1995-11-17,Why can't the most powerful man in the world h...,776,6.530,/yObOAYFIHXHkFPQ3jhgkN2ezaD.jpg,/62BnXyJtVEq4WKNSpnPG7QPYYDI.jpg,"['Michael Douglas', 'Annette Bening', 'Martin ...",Rob Reiner,"['new love', 'usa president', 'the white house...",6.580889,4


### Función de Descarga

Creamos la función de descarga especificando el parámetro `language=es-MX` en la URL de la API para obtener los datos traducidos al español de México.

In [3]:
def fetch_movie(movie_id, api_key):
    """
    Function to fetch data for a single movie from TMDB API
    Args:
        movie_id (int): TMDB_ID of the movie
        api_key (str): API key for authentication

    Returns:
        dict: Movie data if request is successful, None if failed
    """
    url = f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={api_key}&language=es-Mx"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            return response.json()
        else:
            return None  # Return None if response code is not 200 (success)

    except requests.exceptions.RequestException as e:
        # Handle any network-related or request errors
        print(f"Error fetching data for movie ID {movie_id}: {e}")
        return None

In [4]:
def fetch_movies(ids, api_key):
    """
    Function to fetch movie data for multiple movie IDs using concurrent requests
    Args:
        ids (list): List of movie IDs to fetch
        api_key (str): API key for authentication

    Returns:
        tuple: A tuple containing two lists:
            - List of successfully fetched movies (as JSON)
            - List of movie IDs for which fetching data failed
    """
    id_errors = []
    movies = []

    # Using ThreadPoolExecutor to send multiple requests concurrently
    with ThreadPoolExecutor(max_workers=10) as executor:
        # Submitting the fetch_movie function to the executor for each movie ID
        futures = {executor.submit(fetch_movie, movie_id, api_key): movie_id for movie_id in ids}

        # Processing results as they complete
        for future in as_completed(futures):
            movie_id = futures[future]
            movie_data = future.result()

            if movie_data:
                movies.append(movie_data)
            else:
                id_errors.append(movie_id)

            # Adding a small sleep time to avoid hitting API rate limits
            time.sleep(0.005)

    return movies, id_errors

### Extracción de IDs y Descarga

Extraemos los `tmdb_id` de todas las películas del catálogo y lanzamos las peticiones concurrentes.

In [5]:
ids = mdf['tmdb_id'].to_list()
ids[:5]

[710, 949, 862, 8844, 9087]

In [6]:
api_key = os.getenv('tmdb_api_key')
movies, id_errors = fetch_movies(ids, api_key)

print(f"Fetched {len(movies)} movies")
print(f"Failed to fetch {len(id_errors)} movies")

Fetched 5000 movies
Failed to fetch 0 movies


In [7]:
fetched = pd.DataFrame(data=movies)
fetched.head().transpose()

,0,1,2,3,4
adult,False,False,False,False,False
backdrop_path,/qSxeCfWUUyht9hZgaaYmtPtTkw2.jpg,/zAaFQHZV23SJgSwRBvW5PeD219H.jpg,/62BnXyJtVEq4WKNSpnPG7QPYYDI.jpg,/xKsnZDERG1dk95wuZ5q9iks3OL3.jpg,/3Rfvhy1Nl6sSGJwyjb0QiZzZYlB.jpg
belongs_to_collection,"{'id': 495527, 'name': 'Jumanji - Colección', ...",None,None,"{'id': 1048282, 'name': 'Heat Collection', 'po...","{'id': 10194, 'name': 'Toy Story - Colección',..."
budget,65000000,44000000,62000000,60000000,30000000
genres,"[{'id': 12, 'name': 'Aventura'}, {'id': 14, 'n...","[{'id': 18, 'name': 'Drama'}, {'id': 36, 'name...","[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...","[{'id': 80, 'name': 'Crimen'}, {'id': 18, 'nam...","[{'id': 10751, 'name': 'Familia'}, {'id': 35, ..."
homepage,,,,,
id,8844,10858,9087,949,862
imdb_id,tt0113497,tt0113987,tt0112346,tt0113277,tt0114709
origin_country,[US],[US],[US],[US],[US]
original_language,en,en,en,en,en


### Selección de Columnas Relevantes

De la respuesta de la API, seleccionamos únicamente los campos que necesitamos en español. Los demás campos (director, keywords, score, etc.) los traemos del archivo en inglés.

In [8]:
# Select only the relevant columns
fetched = fetched[['id', 'title', 'genres', 'overview', 
           'release_date', 'runtime', 'tagline',  'vote_average', 
           'popularity', 'vote_count', 'poster_path', 'backdrop_path']]

fetched.head().transpose()

,0,1,2,3,4
id,8844,10858,9087,949,862
title,Jumanji,Nixon,Mi querido presidente,Fuego contra Fuego,Toy Story
genres,"[{'id': 12, 'name': 'Aventura'}, {'id': 14, 'n...","[{'id': 18, 'name': 'Drama'}, {'id': 36, 'name...","[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...","[{'id': 80, 'name': 'Crimen'}, {'id': 18, 'nam...","[{'id': 10751, 'name': 'Familia'}, {'id': 35, ..."
overview,Alan Parrish queda atrapado durante 25 años en...,Richard Nixon fue uno de los presidentes más c...,"Andrew Shepherd (Michael Douglas), el Presiden...",Neil McCauley es un experto ladrón. Su filosof...,Un muñeco vaquero se siente profundamente amen...
release_date,1995-12-15,1995-12-22,1995-11-17,1995-12-15,1995-11-22
runtime,104,192,114,170,81
tagline,Cuando oigas los tambores... algo malo ocurre.,,¿Por qué el hombre más poderoso del mundo no p...,Una saga de crimen de Los Ángeles,Bienvenido a un mundo asombroso donde los jugu...
vote_average,7.2,6.9,6.53,7.9,7.971
popularity,3.3337,2.6466,2.6693,15.136,20.4112
vote_count,11207,391,776,8226,19723


In [9]:
mdf.columns

Index(['movielens_id', 'tmdb_id', 'title', 'genres', 'overview', 'runtime',
       'release_date', 'tagline', 'vote_count', 'vote_average', 'poster_path',
       'backdrop_path', 'cast', 'director', 'keywords', 'score', 'movie_id'],
      dtype='object')

### Merge con `movies_final.csv`

Hacemos un merge para combinar los campos en español (descargados ahora) con los campos que ya teníamos en inglés (director, actores, palabras clave, score, IDs internos).

In [10]:
# Select the missing columns from movies_final.csv
mdf = mdf[['tmdb_id', 'movielens_id', 'movie_id', 'cast', 'director', 'keywords', 'score']]
mdf.head()

,tmdb_id,movielens_id,movie_id,cast,director,keywords,score
0,710,10,0,"['Pierce Brosnan', 'Sean Bean', 'Izabella Scor...",Martin Campbell,"['computer virus', 'cuba', 'falsely accused', ...",6.904242
1,949,6,1,"['Al Pacino', 'Robert De Niro', 'Val Kilmer']",Michael Mann,"['robbery', 'chase', 'obsession', 'detective',...",7.925015
2,862,1,2,"['Tom Hanks', 'Tim Allen', 'Don Rickles']",John Lasseter,"['rescue', 'friendship', 'mission', 'jealousy'...",7.968359
3,8844,2,3,"['Robin Williams', 'Kirsten Dunst', 'Bradley P...",Joe Johnston,"['giant insect', 'board game', 'disappearance'...",7.199878
4,9087,11,4,"['Michael Douglas', 'Annette Bening', 'Martin ...",Rob Reiner,"['new love', 'usa president', 'the white house...",6.580889


In [11]:
# Rename API 'id' to match our convention before merge
fetched = fetched.rename(columns={'id': 'tmdb_id'})
spanish_df = pd.merge(fetched, mdf, on='tmdb_id')
spanish_df.head().transpose()

,0,1,2,3,4
tmdb_id,8844,10858,9087,949,862
title,Jumanji,Nixon,Mi querido presidente,Fuego contra Fuego,Toy Story
genres,"[{'id': 12, 'name': 'Aventura'}, {'id': 14, 'n...","[{'id': 18, 'name': 'Drama'}, {'id': 36, 'name...","[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...","[{'id': 80, 'name': 'Crimen'}, {'id': 18, 'nam...","[{'id': 10751, 'name': 'Familia'}, {'id': 35, ..."
overview,Alan Parrish queda atrapado durante 25 años en...,Richard Nixon fue uno de los presidentes más c...,"Andrew Shepherd (Michael Douglas), el Presiden...",Neil McCauley es un experto ladrón. Su filosof...,Un muñeco vaquero se siente profundamente amen...
release_date,1995-12-15,1995-12-22,1995-11-17,1995-12-15,1995-11-22
runtime,104,192,114,170,81
tagline,Cuando oigas los tambores... algo malo ocurre.,,¿Por qué el hombre más poderoso del mundo no p...,Una saga de crimen de Los Ángeles,Bienvenido a un mundo asombroso donde los jugu...
vote_average,7.2,6.9,6.53,7.9,7.971
popularity,3.3337,2.6466,2.6693,15.136,20.4112
vote_count,11207,391,776,8226,19723


In [12]:
# Sort the columns by relevance
final_df = spanish_df[['movielens_id', 'movie_id', 'tmdb_id', 'title', 'genres',
       'overview', 'score', 'release_date', 'runtime', 'tagline',
       'vote_average', 'vote_count', 'popularity', 'cast', 'director',
       'keywords', 'poster_path', 'backdrop_path'
       ]]
# Sort the movies by internal id
final_df = final_df.sort_values(by='movie_id', ascending=True)
final_df.head()

,movielens_id,movie_id,tmdb_id,title,genres,overview,score,release_date,runtime,tagline,vote_average,vote_count,popularity,cast,director,keywords,poster_path,backdrop_path
7,10,0,710,"007: Goldeneye, el Regreso del Agente","[{'id': 12, 'name': 'Aventura'}, {'id': 28, 'n...","Estando de vacaciones, Bond conoce a la bella ...",6.904242,1995-11-16,130,"Sin límites, no conoce el miedo, es único",6.900,4278,9.0518,"['Pierce Brosnan', 'Sean Bean', 'Izabella Scor...",Martin Campbell,"['computer virus', 'cuba', 'falsely accused', ...",/9bVk0jq07Sgz7axpP68UtkvZF4C.jpg,/fIWsCpYR9iGDMSbMTSAzy8L7Kg5.jpg
3,6,1,949,Fuego contra Fuego,"[{'id': 80, 'name': 'Crimen'}, {'id': 18, 'nam...",Neil McCauley es un experto ladrón. Su filosof...,7.925015,1995-12-15,170,Una saga de crimen de Los Ángeles,7.900,8226,15.1360,"['Al Pacino', 'Robert De Niro', 'Val Kilmer']",Michael Mann,"['robbery', 'chase', 'obsession', 'detective',...",/c1qMPl0takx8I4JFxDXMAEEOKXg.jpg,/xKsnZDERG1dk95wuZ5q9iks3OL3.jpg
4,1,2,862,Toy Story,"[{'id': 10751, 'name': 'Familia'}, {'id': 35, ...",Un muñeco vaquero se siente profundamente amen...,7.968359,1995-11-22,81,Bienvenido a un mundo asombroso donde los jugu...,7.971,19723,20.4112,"['Tom Hanks', 'Tim Allen', 'Don Rickles']",John Lasseter,"['rescue', 'friendship', 'mission', 'jealousy'...",/y59d8PjuFSgJJN4VRS7jDoRgSM9.jpg,/3Rfvhy1Nl6sSGJwyjb0QiZzZYlB.jpg
0,2,3,8844,Jumanji,"[{'id': 12, 'name': 'Aventura'}, {'id': 14, 'n...",Alan Parrish queda atrapado durante 25 años en...,7.199878,1995-12-15,104,Cuando oigas los tambores... algo malo ocurre.,7.200,11207,3.3337,"['Robin Williams', 'Kirsten Dunst', 'Bradley P...",Joe Johnston,"['giant insect', 'board game', 'disappearance'...",/m67M2wXLKblJLuJlvF7qaGrGfCa.jpg,/qSxeCfWUUyht9hZgaaYmtPtTkw2.jpg
2,11,4,9087,Mi querido presidente,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...","Andrew Shepherd (Michael Douglas), el Presiden...",6.580889,1995-11-17,114,¿Por qué el hombre más poderoso del mundo no p...,6.530,776,2.6693,"['Michael Douglas', 'Annette Bening', 'Martin ...",Rob Reiner,"['new love', 'usa president', 'the white house...",/ir4oGHypSMliG4rnxo2CTp6y1A0.jpg,/62BnXyJtVEq4WKNSpnPG7QPYYDI.jpg


### Verificación de Nulos

Comprobamos que los campos críticos (título, `tmdb_id`, posters) no tengan valores nulos.

In [15]:
# Check if there is an important null field
final_df[['title', 'tmdb_id', 'poster_path', 'backdrop_path']].isna().sum()

title            0
tmdb_id          0
poster_path      0
backdrop_path    0
dtype: int64

### Guardado del Dataset Final en Español

Guardamos el dataset completo en español. Este archivo será utilizado por el script de base de datos (`database/create_script.py`) para generar el SQL de inserción de películas en la BD.

In [16]:
final_df.to_csv('../data/processed/movies_final_spanish.csv', index=False)

## Resumen del Notebook

En este notebook se descargaron los metadatos de las 5,000 películas del catálogo en español desde la API de TMDB (con parámetro `language=es-MX`).

**Acciones realizadas:**
- Descarga de título, sinopsis, eslogan y géneros en español para cada película
- Merge con `movies_final.csv` para añadir director, actores, keywords, score e IDs internos
- Verificación de nulos en campos críticos

**Resultado:** Dataset con las 5,000 películas con contenido en español

**Output generado:** `ml/data/processed/movies_final_spanish.csv`